In [8]:
import cv2
import numpy as np
import os
from rapidocr_onnxruntime import RapidOCR


def cluster_boundaries(coords, threshold=15):
    """Groups coordinates that are very close to each other to form a logical grid."""
    if not coords:
        return []
    coords = sorted(coords)
    clusters = []
    current_cluster = [coords[0]]

    for c in coords[1:]:
        if c - current_cluster[-1] <= threshold:
            current_cluster.append(c)
        else:
            clusters.append(int(np.mean(current_cluster)))
            current_cluster = [c]

    if current_cluster:
        clusters.append(int(np.mean(current_cluster)))

    return clusters


def find_grid_lines(line_mask, axis, min_fraction, cluster_threshold=15):
    """
    Finds boundary positions by looking for lines that span at least
    `min_fraction` of the table's width/height.

    IMPORTANT: min_fraction must be LOW (e.g. 0.15) here, not 0.5. A line that
    borders a merged/spanned cell only runs across the *unmerged* columns, so
    it can legitimately cover well under 50% of the full table width. Using a
    high threshold silently drops those partial-width separators and merges
    real rows/columns together. The morphological opening step already uses a
    long kernel, so short noise segments (e.g. from text strokes) are removed
    before this function ever sees them - a low fraction here is safe.

    axis=0 -> looking for HORIZONTAL lines (row boundaries), scan by row
    axis=1 -> looking for VERTICAL lines (col boundaries), scan by column
    """
    h, w = line_mask.shape
    if axis == 0:
        coverage = (line_mask > 0).sum(axis=1) / float(w)
    else:
        coverage = (line_mask > 0).sum(axis=0) / float(h)

    candidate_positions = np.where(coverage >= min_fraction)[0].tolist()
    return cluster_boundaries(candidate_positions, threshold=cluster_threshold)


def local_border_exists(mask, y0, y1, x0, x1, axis, tol=4, local_fraction=0.5):
    """
    Checks whether a line actually exists LOCALLY along one edge of a specific
    fine grid cell (as opposed to the global full-width/height check used to
    build the candidate grid). This is what lets us tell a real row/column
    boundary apart from a merged cell's missing internal border.

    axis=0 -> checking a HORIZONTAL border (top edge of a cell): look in a
              thin horizontal strip around y0, spanning [x0, x1)
    axis=1 -> checking a VERTICAL border (left edge of a cell): look in a
              thin vertical strip around x0, spanning [y0, y1)
    """
    h, w = mask.shape
    if axis == 0:
        y_lo, y_hi = max(0, y0 - tol), min(h, y0 + tol)
        strip = mask[y_lo:y_hi, x0:x1]
        span = max(1, x1 - x0)
    else:
        x_lo, x_hi = max(0, x0 - tol), min(w, x0 + tol)
        strip = mask[y0:y1, x_lo:x_hi]
        span = max(1, y1 - y0)

    if strip.size == 0:
        return False

    # max coverage along the line direction (robust to the strip being a
    # couple pixels off from the true line center)
    if axis == 0:
        coverage = (strip > 0).sum(axis=1).max() / float(span)
    else:
        coverage = (strip > 0).sum(axis=0).max() / float(span)

    return coverage >= local_fraction


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb


def extract_table(image_path):
    # 1. Load Image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not load image at {image_path}")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 2. AUTO-DETECT DARK OR LIGHT MODE
    is_dark_mode = np.median(gray) < 127
    bg_color = 0 if is_dark_mode else 255

    # 3. ADD PADDING (Crucial for tables touching the image edges)
    pad_size = 20
    gray = cv2.copyMakeBorder(gray, pad_size, pad_size, pad_size, pad_size,
                               cv2.BORDER_CONSTANT, value=bg_color)

    # Thresholding
    if is_dark_mode:
        _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    else:
        _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)

    # 4. Detect Horizontal and Vertical Lines
    kernel_len = max(40, gray.shape[1] // 40)
    ver_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, kernel_len))
    hor_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_len, 1))

    vertical_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, ver_kernel, iterations=2)
    horizontal_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, hor_kernel, iterations=2)

    # 5. LINE ERASER (Deletes lines from the image so RapidOCR doesn't read them as letters)
    line_dilate_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    thick_skeleton = cv2.dilate(cv2.bitwise_or(vertical_lines, horizontal_lines),
                                 line_dilate_kernel, iterations=1)

    clean_gray = gray.copy()
    clean_gray[thick_skeleton > 0] = bg_color

    # 6. FIND CANDIDATE GRID LINES VIA PROJECTION.
    #    min_fraction is intentionally LOW so that lines bordering a merged
    #    cell (which only run across the un-merged columns/rows) are still
    #    picked up as real boundaries. See find_grid_lines() docstring.
    row_boundaries = find_grid_lines(horizontal_lines, axis=0, min_fraction=0.15)
    col_boundaries = find_grid_lines(vertical_lines, axis=1, min_fraction=0.15)

    num_rows = max(0, len(row_boundaries) - 1)
    num_cols = max(0, len(col_boundaries) - 1)

    if num_rows == 0 or num_cols == 0:
        raise ValueError("Could not detect a table grid in this image.")

    # 7. DETERMINE MERGES (rowspan/colspan) BY CHECKING LOCAL BORDERS.
    #    For every fine grid cell, check whether it actually has a top border
    #    and a left border where it individually sits. If a border is
    #    missing, the cell is merged with its neighbour above/left.
    uf = UnionFind(num_rows * num_cols)

    def idx(r, c):
        return r * num_cols + c

    for r in range(num_rows):
        y0, y1 = row_boundaries[r], row_boundaries[r + 1]
        for c in range(num_cols):
            x0, x1 = col_boundaries[c], col_boundaries[c + 1]

            if r > 0:
                has_top = local_border_exists(horizontal_lines, y0, y1, x0, x1, axis=0)
                if not has_top:
                    uf.union(idx(r, c), idx(r - 1, c))

            if c > 0:
                has_left = local_border_exists(vertical_lines, y0, y1, x0, x1, axis=1)
                if not has_left:
                    uf.union(idx(r, c), idx(r, c - 1))

    # 8. GROUP FINE CELLS INTO MERGED REGIONS, COMPUTE BOUNDING BOXES.
    groups = {}
    for r in range(num_rows):
        for c in range(num_cols):
            root = uf.find(idx(r, c))
            groups.setdefault(root, []).append((r, c))

    ocr_engine = RapidOCR()
    margin = 3
    upscale_factor = 2

    def ocr_crop(y0, y1, x0, x1):
        crop_img = clean_gray[y0 + margin: y1 - margin, x0 + margin: x1 - margin]
        if crop_img.shape[0] <= 5 or crop_img.shape[1] <= 5:
            return ""
        if is_dark_mode:
            crop_img = cv2.bitwise_not(crop_img)
        crop_img = cv2.resize(crop_img, None, fx=upscale_factor, fy=upscale_factor,
                               interpolation=cv2.INTER_CUBIC)
        crop_img = cv2.copyMakeBorder(crop_img, 15, 15, 15, 15,
                                       cv2.BORDER_CONSTANT, value=255)
        crop_bgr = cv2.cvtColor(crop_img, cv2.COLOR_GRAY2BGR)
        ocr_result, _ = ocr_engine(crop_bgr)
        if not ocr_result:
            return ""
        ocr_result = sorted(ocr_result, key=lambda line: line[0][0][1])
        return "\n".join([line[1] for line in ocr_result]).strip()

    # cell_info[(r,c)] = None (covered by a span, skip) or dict with
    # rowspan/colspan/text (anchor cell, top-left of its group)
    cell_info = {}
    for root, members in groups.items():
        rs = [m[0] for m in members]
        cs = [m[1] for m in members]
        min_r, max_r = min(rs), max(rs)
        min_c, max_c = min(cs), max(cs)

        y0, y1 = row_boundaries[min_r], row_boundaries[max_r + 1]
        x0, x1 = col_boundaries[min_c], col_boundaries[max_c + 1]
        text = ocr_crop(y0, y1, x0, x1)

        cell_info[(min_r, min_c)] = {
            "rowspan": max_r - min_r + 1,
            "colspan": max_c - min_c + 1,
            "text": text,
        }
        for (r, c) in members:
            if (r, c) != (min_r, min_c):
                cell_info[(r, c)] = None  # covered by the span above

    # 9. Convert to Clean HTML using rowspan/colspan.
    html = [
        "<table border='1' style='border-collapse: collapse; font-family: sans-serif;'>",
        "<style> td { padding: 8px; text-align: left; } th { padding: 8px; text-align: left; } </style>",
    ]

    for r in range(num_rows):
        html.append("  <tr>")
        tag = "th" if r == 0 else "td"
        for c in range(num_cols):
            info = cell_info.get((r, c))
            if info is None:
                continue  # covered by a rowspan/colspan from an earlier cell
            attrs = ""
            if info["rowspan"] > 1:
                attrs += f" rowspan='{info['rowspan']}'"
            if info["colspan"] > 1:
                attrs += f" colspan='{info['colspan']}'"
            safe_text = info["text"].replace('\n', '<br>')
            html.append(f"    <{tag}{attrs}>{safe_text}</{tag}>")
        html.append("  </tr>")
    html.append("</table>")

    return "\n".join(html)


def save_html_unique(html_content, base_filepath="output.html"):
    directory, filename = os.path.split(base_filepath)
    name, ext = os.path.splitext(filename)

    final_filepath = base_filepath
    counter = 1

    while os.path.exists(final_filepath):
        new_filename = f"{name}_{counter}{ext}"
        final_filepath = os.path.join(directory, new_filename)
        counter += 1

    with open(final_filepath, "w", encoding="utf-8") as f:
        f.write(html_content)

    print(f"Success! File saved as: {final_filepath}")
    return final_filepath


if __name__ == "__main__":
    image_path = r"C:\Users\Dell\Pictures\Screenshots\Screenshot 2026-08-06 001025.png"

    html_table = extract_table(image_path)
    save_html_unique(html_table, r"C:\Users\Dell\Downloads\table_output.html")

Success! File saved as: C:\Users\Dell\Downloads\table_output_11.html


In [11]:
"""
Convert an HTML <table> (with rowspan/colspan) into a Word table, using only
python-docx + bs4 - no pandoc, no shelling out.

The core problem this solves: HTML rowspan/colspan and python-docx's
cell.merge() both describe "which cells belong together", but they don't
speak the same coordinate system. HTML lets a row simply have fewer <td>s
when a cell above is still spanning down into it. python-docx has no such
shorthand - every row of the table must have a real cell in every column,
and merges are applied *after* the fact by merging two corner cells.

So the approach is:
  1. Parse the HTML into a dense (row, col) grid, exactly like a browser
     would - every grid position is filled in, including positions "covered"
     by a spanning cell from above/left. This is the same algorithm browsers
     use internally to lay out tables.
  2. Create a docx table of that grid's exact dimensions.
  3. Write text only into each span's origin (top-left) cell.
  4. For every span with rowspan>1 or colspan>1, call cell.merge() between
     its top-left and bottom-right grid positions - this is what actually
     produces <w:vMerge>/<w:gridSpan> in the docx XML.
"""

from bs4 import BeautifulSoup
from docx import Document
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from docx.shared import Pt


def _set_cell_borders(cell, sides=("top", "bottom", "left", "right"),
                       size=4, color="000000"):
    """python-docx has no built-in API for cell borders - they live in
    <w:tcPr><w:tcBorders> and have to be added at the XML level."""
    tcPr = cell._tc.get_or_add_tcPr()
    tcBorders = tcPr.find(qn('w:tcBorders'))
    if tcBorders is None:
        tcBorders = OxmlElement('w:tcBorders')
        tcPr.append(tcBorders)
    for side in sides:
        tag = f'w:{side}'
        el = tcBorders.find(qn(tag))
        if el is None:
            el = OxmlElement(tag)
            tcBorders.append(el)
        el.set(qn('w:val'), 'single')
        el.set(qn('w:sz'), str(size))
        el.set(qn('w:space'), '0')
        el.set(qn('w:color'), color)


def _parse_html_table_to_grid(html_table: str):
    """
    Turns the HTML table into a dense grid: grid[r][c] = {
        'text': str, 'is_origin': bool, 'r0','c0','r1','c1': span bounds,
        'header': bool
    }
    Cells covered by a span (not the origin) just point at the same dict as
    their origin cell, so later code can tell "this grid slot belongs to the
    span that started at (r0,c0)".
    """
    soup = BeautifulSoup(html_table, "html.parser")
    table = soup.find("table")
    if table is None:
        raise ValueError("No <table> found in the given HTML.")

    rows = table.find_all("tr", recursive=True)
    # rowspan tracker: for each column, how many more rows a previous cell
    # still occupies, and which cell dict it belongs to.
    pending = {}  # col_index -> (remaining_rows, cell_dict)
    grid = []
    num_cols = 0

    for r_idx, tr in enumerate(rows):
        grid.append({})
        c_idx = 0
        cells = tr.find_all(["td", "th"], recursive=False)
        cell_iter = iter(cells)
        current_cell = next(cell_iter, None)

        while current_cell is not None or c_idx in pending:
            # fill in any column still covered by a rowspan from above
            if c_idx in pending:
                remaining, cell_dict = pending[c_idx]
                grid[r_idx][c_idx] = cell_dict
                if remaining - 1 > 0:
                    pending[c_idx] = (remaining - 1, cell_dict)
                else:
                    del pending[c_idx]
                c_idx += 1
                continue

            if current_cell is None:
                break

            rowspan = int(current_cell.get("rowspan", 1) or 1)
            colspan = int(current_cell.get("colspan", 1) or 1)
            text = current_cell.get_text(separator="\n", strip=True)
            is_header = current_cell.name == "th"

            cell_dict = {
                "text": text,
                "is_origin": True,
                "r0": r_idx, "c0": c_idx,
                "r1": r_idx + rowspan - 1,
                "c1": c_idx + colspan - 1,
                "header": is_header,
            }

            for cc in range(c_idx, c_idx + colspan):
                grid[r_idx][cc] = cell_dict
                if rowspan > 1:
                    pending[cc] = (rowspan - 1, cell_dict)

            c_idx += colspan
            current_cell = next(cell_iter, None)

        num_cols = max(num_cols, c_idx)

    num_rows = len(grid)
    # normalize: make sure every row dict has an entry for every column
    # (can happen if a row's spans don't reach the table's max width)
    for r in range(num_rows):
        for c in range(num_cols):
            if c not in grid[r]:
                grid[r][c] = {
                    "text": "", "is_origin": True,
                    "r0": r, "c0": c, "r1": r, "c1": c, "header": False,
                }

    return grid, num_rows, num_cols


def html_table_to_docx(html_table: str, output_path: str,
                        add_borders: bool = True,
                        bold_header: bool = True,
                        font_size_pt: float = 10,
                        autofit: bool = True) -> str:
    """
    Convert an HTML table string into a .docx file containing a single Word
    table with the exact same row/column structure and merged cells.

    Args:
        html_table: HTML containing a <table> (a full page or a fragment,
            e.g. what extract_table() in extract_table_fixed.py returns).
        output_path: path to write the .docx to.
        add_borders: draw visible single-line borders on every cell.
        bold_header: bold the text in <th> cells.
        font_size_pt: font size for all cell text.
        autofit: let Word auto-fit column widths to content.

    Returns:
        output_path
    """
    grid, num_rows, num_cols = _parse_html_table_to_grid(html_table)
    if num_rows == 0 or num_cols == 0:
        raise ValueError("Parsed table has no rows/columns.")

    doc = Document()
    table = doc.add_table(rows=num_rows, cols=num_cols)
    table.style = "Table Grid" if add_borders else "Normal Table"
    if autofit:
        table.autofit = True

    # 1. Write text only into each span's origin cell.
    written = set()
    for r in range(num_rows):
        for c in range(num_cols):
            info = grid[r][c]
            key = (info["r0"], info["c0"])
            if key in written:
                continue
            written.add(key)

            cell = table.cell(info["r0"], info["c0"])
            cell.text = ""  # clear default empty paragraph text run
            paragraph = cell.paragraphs[0]
            run = paragraph.add_run(info["text"])
            run.font.size = Pt(font_size_pt)
            if bold_header and info["header"]:
                run.bold = True

    # 2. Apply merges for every span with rowspan/colspan > 1 (once per span).
    merged_already = set()
    for r in range(num_rows):
        for c in range(num_cols):
            info = grid[r][c]
            span_key = (info["r0"], info["c0"], info["r1"], info["c1"])
            if span_key in merged_already:
                continue
            if info["r1"] > info["r0"] or info["c1"] > info["c0"]:
                top_left = table.cell(info["r0"], info["c0"])
                bottom_right = table.cell(info["r1"], info["c1"])
                top_left.merge(bottom_right)
            merged_already.add(span_key)

    # 3. Borders (python-docx's "Table Grid" style already gives borders,
    #    but we set them explicitly too so the result doesn't depend on the
    #    style being available/rendered a certain way in every Word version).
    if add_borders:
        for row in table.rows:
            for cell in row.cells:
                _set_cell_borders(cell)

    doc.save(output_path)
    return output_path


if __name__ == "__main__":
    import sys
    html_file = r"C:\Users\Dell\Downloads\table_output_10.html"
    with open(html_file, "r", encoding="utf-8") as f:
        html = f.read()
    out = html_table_to_docx(html, r"C:\Users\Dell\Downloads\output.docx")
    print(f"Saved: {out}")

Saved: C:\Users\Dell\Downloads\output.docx


In [3]:
import cv2
import numpy as np
import os

def crop_table_with_padding(image_path, output_path="cropped_table.png", padding=20):
    """
    Detects a table in an image, crops it exactly to its borders, 
    and adds a uniform background padding around it.
    
    :param padding: The number of pixels to add as margin (20px is roughly 2-3pt on standard screens)
    """
    # 1. Load the image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Cannot read image at: {image_path}")
        
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 2. Auto-detect Dark/Light mode to pick the right background color for our padding
    is_dark_mode = np.median(gray) < 127
    # Use dark gray for dark mode, pure white for light mode
    bg_color = [30, 30, 30] if is_dark_mode else [255, 255, 255] 
    
    # 3. Threshold the image
    if is_dark_mode:
        _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
    else:
        _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
        
    # 4. Detect Horizontal and Vertical Lines
    kernel_len = max(20, img.shape[1] // 50)
    ver_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, kernel_len))
    hor_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_len, 1))
    
    vertical_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, ver_kernel, iterations=2)
    horizontal_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, hor_kernel, iterations=2)
    
    # Combine lines to create the grid skeleton
    table_skeleton = cv2.bitwise_or(vertical_lines, horizontal_lines)
    
    # 5. Find contours of the grid
    contours, _ = cv2.findContours(table_skeleton, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    valid_boxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        # Filter out tiny specks of noise; only keep distinct lines/boxes
        if w > 20 or h > 20: 
            valid_boxes.append((x, y, w, h))
            
    if not valid_boxes:
        print("No table structure could be detected in the image.")
        return None
        
    # 6. Find the absolute extreme boundaries of the entire table structure
    min_x = min([b[0] for b in valid_boxes])
    min_y = min([b[1] for b in valid_boxes])
    max_x = max([b[0] + b[2] for b in valid_boxes])
    max_y = max([b[1] + b[3] for b in valid_boxes])
    
    # 7. Crop the image EXACTLY to the table's outer borders
    exact_crop = img[min_y:max_y, min_x:max_x]
    
    # 8. Add the uniform padding (margin) around the cropped table
    padded_table = cv2.copyMakeBorder(
        exact_crop, 
        padding, padding, padding, padding, 
        cv2.BORDER_CONSTANT, 
        value=bg_color
    )
    
    # 9. Ensure a unique filename and save
    directory, filename = os.path.split(output_path)
    name, ext = os.path.splitext(filename)
    final_path = output_path
    counter = 1
    
    while os.path.exists(final_path):
        final_path = os.path.join(directory, f"{name}_{counter}{ext}")
        counter += 1
        
    cv2.imwrite(final_path, padded_table)
    print(f"Success! Cropped and padded table saved to: {final_path}")
    
    return final_path

if __name__ == "__main__":
    input_image = r"C:\Users\Dell\Pictures\Screenshots\Screenshot 2026-08-06 003306.png"
    
    # 20 pixels of padding is an excellent buffer for OCR/HTML parsing tools
    output_image = crop_table_with_padding(input_image, padding=20)

Success! Cropped and padded table saved to: cropped_table_1.png


In [1]:
print('hi')

hi
